# TF-IDF Model Comparison - PubMedQA Medical Question Answering

## 5 Models Compared:

1. **Logistic Regression** - Linear baseline
2. **Linear SVM** - Margin-based (expected best)
3. **XGBoost** - Gradient boosting
4. **Random Forest** - Bagging ensemble
5. **Multi-Layer Perceptron (MLP)** - Neural network



## 1. Setup and Installation

In [1]:
# !pip install -q datasets transformers scikit-learn xgboost pandas numpy matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import time
import warnings
from pathlib import Path

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, precision_recall_fscore_support
)

import xgboost as xgb

from datasets import load_dataset

warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.style.use('seaborn-v0_8-paper')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11


## 2. Data Loading from HuggingFace

In [2]:
dataset = load_dataset('pubmed_qa', 'pqa_labeled', split='train')

print(f"Loaded {len(dataset)} examples")
print(f"\nDataset features: {list(dataset.features.keys())}")

Loaded 1000 examples

Dataset features: ['pubid', 'question', 'context', 'long_answer', 'final_decision']


In [3]:

records = []

for idx, item in enumerate(dataset):
    abstract = ' '.join(item['context']['contexts'])
    
    records.append({
        'qid': str(idx),
        'question': item['question'],
        'abstract': abstract,
        'long_answer': item['long_answer'],
        'label': item['final_decision'].lower(),
        'question_length': len(item['question'].split()),
        'abstract_length': len(abstract.split())
    })

df = pd.DataFrame(records)


print(f"  Shape: {df.shape}")
print(f"  Columns: {list(df.columns)}")
df[['question', 'label', 'question_length', 'abstract_length']].head()

  Shape: (1000, 7)
  Columns: ['qid', 'question', 'abstract', 'long_answer', 'label', 'question_length', 'abstract_length']


,question,label,question_length,abstract_length
0,Do mitochondria play a role in remodelling lac...,yes,14,251
1,Landolt C and snellen e acuity: differences in...,no,10,245
2,"Syncope during bathing in infants, a pediatric...",yes,11,173
3,Are the long-term results of the transanal pul...,no,15,223
4,Can tailored interventions increase mammograph...,yes,9,194


## 3. Data Preprocessing and Splitting

In [4]:
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s.,;:?!-]', ' ', text)
    return text.lower().strip()

def combine_texts(questions, abstracts):
    return [f"{q} {q} {a}" for q, a in zip(questions, abstracts)]


In [5]:
TRAIN_SIZE = 0.7
TEST_SIZE = 0.3

train_df, test_df = train_test_split(
    df,
    train_size=TRAIN_SIZE,
    stratify=df['label'],
    random_state=RANDOM_SEED
)


train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f" Train set: {len(train_df)} examples ({len(train_df)/len(df)*100:.1f}%)")
print(f" Test set:  {len(test_df)} examples ({len(test_df)/len(df)*100:.1f}%)")


print(f"\nTrain set:")
train_dist = train_df['label'].value_counts(normalize=True) * 100
for label in ['yes', 'no', 'maybe']:
    print(f"  {label:8s}: {train_dist.get(label, 0):5.2f}%")

print(f"\nTest set:")
test_dist = test_df['label'].value_counts(normalize=True) * 100
for label in ['yes', 'no', 'maybe']:
    print(f"  {label:8s}: {test_dist.get(label, 0):5.2f}%")

 Train set: 700 examples (70.0%)
 Test set:  300 examples (30.0%)

Train set:
  yes     : 55.14%
  no      : 33.86%
  maybe   : 11.00%

Test set:
  yes     : 55.33%
  no      : 33.67%
  maybe   : 11.00%


## 4. TF-IDF Feature Extraction

In [6]:
train_questions = [preprocess_text(q) for q in train_df['question']]
train_abstracts = [preprocess_text(a) for a in train_df['abstract']]
train_texts = combine_texts(train_questions, train_abstracts)

test_questions = [preprocess_text(q) for q in test_df['question']]
test_abstracts = [preprocess_text(a) for a in test_df['abstract']]
test_texts = combine_texts(test_questions, test_abstracts)


vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),      
    max_features=10000,      
    min_df=2,                
    max_df=0.95,             
    use_idf=True,            
    sublinear_tf=True,      
    strip_accents='unicode'
)

X_train = vectorizer.fit_transform(train_texts)
X_test = vectorizer.transform(test_texts)

print(f"\n Feature matrix shape: {X_train.shape}")
print(f"  - Samples: {X_train.shape[0]}")
print(f"  - Features: {X_train.shape[1]}")
print(f"  - Vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"  - Sparsity: {(1 - X_train.nnz / (X_train.shape[0] * X_train.shape[1]))*100:.2f}%")

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_df['label'])
y_test = label_encoder.transform(test_df['label'])

print(f"\n Label encoding:")
for i, label in enumerate(label_encoder.classes_):
    print(f"  {label} → {i}")




 Feature matrix shape: (700, 10000)
  - Samples: 700
  - Features: 10000
  - Vocabulary size: 10000
  - Sparsity: 98.43%

 Label encoding:
  maybe → 0
  no → 1
  yes → 2


In [7]:
feature_names = vectorizer.get_feature_names_out()
tfidf_scores = X_train.sum(axis=0).A1
top_indices = tfidf_scores.argsort()[-20:][::-1]

print("\nTop 20 TF-IDF features (by total score):")
for idx in top_indices:
    print(f"  {feature_names[idx]:30s} {tfidf_scores[idx]:8.2f}")


Top 20 TF-IDF features (by total score):
  with                              19.66
  were                              19.15
  patients                          19.11
  was                               18.66
  for                               17.77
  is                                14.00
  in the                            13.58
  of the                            13.50
  or                                13.00
  by                                12.11
  on                                11.67
  from                              11.53
  at                                11.52
  study                             11.22
  between                           11.17
  group                             10.78
  an                                10.67
  as                                10.65
  this                              10.18
  after                              9.94


## 5. Model Training & Evaluation

We'll train 5 different models with hyperparameter tuning and evaluate each.

In [8]:
def evaluate_model(name, model, X_test, y_test, training_time):
    
    
    start_time = time.time()
    y_pred = model.predict(X_test)
    inference_time = (time.time() - start_time) / len(y_test) * 1000
    
    
    try:
        if hasattr(model, 'predict_proba'):
            y_proba = model.predict_proba(X_test)
        elif hasattr(model, 'decision_function'):
            decision = model.decision_function(X_test)
            exp_decision = np.exp(decision - np.max(decision, axis=1, keepdims=True))
            y_proba = exp_decision / np.sum(exp_decision, axis=1, keepdims=True)
        else:
            y_proba = None
    except:
        y_proba = None
    
    
    accuracy = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    
    precision, recall, f1, support = precision_recall_fscore_support(
        y_test, y_pred, average=None
    )
    
    
    y_pred_decoded = label_encoder.inverse_transform(y_pred)
    y_test_decoded = label_encoder.inverse_transform(y_test)
    

    print(f"{name} - Test Results")
    print(f"Accuracy:       {accuracy:.4f}")
    print(f"F1 (macro):     {f1_macro:.4f}")
    print(f"F1 (weighted):  {f1_weighted:.4f}")
    print(f"Training time:  {training_time:.2f}s")
    print(f"Inference:      {inference_time:.3f} ms/sample")
    
    print(f"\nPer-class metrics:")
    for i, label in enumerate(label_encoder.classes_):
        print(f"  {label:8s}: P={precision[i]:.4f}, R={recall[i]:.4f}, F1={f1[i]:.4f}, Support={int(support[i])}")
    
    print(f"\nClassification Report:")
    print(classification_report(y_test_decoded, y_pred_decoded, digits=4))
    
    return {
        'name': name,
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'per_class_f1': dict(zip(label_encoder.classes_, f1)),
        'per_class_precision': dict(zip(label_encoder.classes_, precision)),
        'per_class_recall': dict(zip(label_encoder.classes_, recall)),
        'per_class_support': dict(zip(label_encoder.classes_, support.astype(int))),
        'training_time': training_time,
        'inference_time_ms': inference_time,
        'predictions': y_pred,
        'probabilities': y_proba,
        'confusion_matrix': confusion_matrix(y_test, y_pred)
    }


all_results = {}


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)


### 5.1 Model 1: Logistic Regression

**Hypothesis**: Strong baseline with good interpretability and probability calibration.

In [9]:

lr_params = {
    'C': [0.1, 1.0, 10.0],
    'penalty': ['l2'],
    'solver': ['lbfgs'],
    'max_iter': [1000],
    'class_weight': ['balanced']
}

start_time = time.time()

lr_grid = GridSearchCV(
    LogisticRegression(random_state=RANDOM_SEED),
    lr_params,
    cv=cv,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

lr_grid.fit(X_train, y_train)
training_time = time.time() - start_time

print(f"\n Best CV F1: {lr_grid.best_score_:.4f}")
print(f" Best params: {lr_grid.best_params_}")

lr_results = evaluate_model(
    "Logistic Regression",
    lr_grid.best_estimator_,
    X_test, y_test,
    training_time
)

all_results['Logistic Regression'] = lr_results

Fitting 5 folds for each of 3 candidates, totalling 15 fits

 Best CV F1: 0.4030
 Best params: {'C': 0.1, 'class_weight': 'balanced', 'max_iter': 1000, 'penalty': 'l2', 'solver': 'lbfgs'}
Logistic Regression - Test Results
Accuracy:       0.5333
F1 (macro):     0.4296
F1 (weighted):  0.5166
Training time:  2.24s
Inference:      0.003 ms/sample

Per-class metrics:
  maybe   : P=0.6250, R=0.1515, F1=0.2439, Support=33
  no      : P=0.4100, R=0.4059, F1=0.4080, Support=101
  yes     : P=0.5938, R=0.6867, F1=0.6369, Support=166

Classification Report:
              precision    recall  f1-score   support

       maybe     0.6250    0.1515    0.2439        33
          no     0.4100    0.4059    0.4080       101
         yes     0.5938    0.6867    0.6369       166

    accuracy                         0.5333       300
   macro avg     0.5429    0.4147    0.4296       300
weighted avg     0.5353    0.5333    0.5166       300



### 5.2 Model 2: Linear SVM

**Hypothesis**: Expected best - margin-based learning is optimal for high-dimensional sparse TF-IDF features.

In [10]:
svm_params = {
    'C': [0.1, 1.0, 10.0],
    'penalty': ['l2'],
    'loss': ['squared_hinge'],
    'class_weight': ['balanced'],
    'max_iter': [2000]
}

start_time = time.time()

svm_grid = GridSearchCV(
    LinearSVC(random_state=RANDOM_SEED, dual=False),
    svm_params,
    cv=cv,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

svm_grid.fit(X_train, y_train)
training_time = time.time() - start_time

print(f"\n Best CV F1: {svm_grid.best_score_:.4f}")
print(f" Best params: {svm_grid.best_params_}")

svm_results = evaluate_model(
    "Linear SVM",
    svm_grid.best_estimator_,
    X_test, y_test,
    training_time
)

all_results['Linear SVM'] = svm_results

Fitting 5 folds for each of 3 candidates, totalling 15 fits

 Best CV F1: 0.3846
 Best params: {'C': 10.0, 'class_weight': 'balanced', 'loss': 'squared_hinge', 'max_iter': 2000, 'penalty': 'l2'}
Linear SVM - Test Results
Accuracy:       0.5433
F1 (macro):     0.3702
F1 (weighted):  0.5054
Training time:  0.31s
Inference:      0.003 ms/sample

Per-class metrics:
  maybe   : P=0.2500, R=0.0303, F1=0.0541, Support=33
  no      : P=0.4390, R=0.3564, F1=0.3934, Support=101
  yes     : P=0.5888, R=0.7590, F1=0.6632, Support=166

Classification Report:
              precision    recall  f1-score   support

       maybe     0.2500    0.0303    0.0541        33
          no     0.4390    0.3564    0.3934       101
         yes     0.5888    0.7590    0.6632       166

    accuracy                         0.5433       300
   macro avg     0.4259    0.3819    0.3702       300
weighted avg     0.5011    0.5433    0.5054       300



### 5.3 Model 3: XGBoost

**Hypothesis**: Captures feature interactions - may excel on complex cases requiring multi-term reasoning.

In [11]:
xgb_params = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.1, 0.3],
    'subsample': [0.8],
    'colsample_bytree': [0.8]
}

start_time = time.time()

xgb_grid = GridSearchCV(
    xgb.XGBClassifier(
        random_state=RANDOM_SEED,
        n_jobs=-1,
        tree_method='hist',
        eval_metric='mlogloss'
    ),
    xgb_params,
    cv=cv,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

xgb_grid.fit(X_train, y_train)
training_time = time.time() - start_time

print(f"\n Best CV F1: {xgb_grid.best_score_:.4f}")
print(f" Best params: {xgb_grid.best_params_}")

xgb_results = evaluate_model(
    "XGBoost",
    xgb_grid.best_estimator_,
    X_test, y_test,
    training_time
)

all_results['XGBoost'] = xgb_results

Fitting 5 folds for each of 12 candidates, totalling 60 fits

 Best CV F1: 0.3847
 Best params: {'colsample_bytree': 0.8, 'learning_rate': 0.3, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}
XGBoost - Test Results
Accuracy:       0.5100
F1 (macro):     0.3308
F1 (weighted):  0.4720
Training time:  42.53s
Inference:      0.008 ms/sample

Per-class metrics:
  maybe   : P=0.0000, R=0.0000, F1=0.0000, Support=33
  no      : P=0.3778, R=0.3366, F1=0.3560, Support=101
  yes     : P=0.5721, R=0.7169, F1=0.6364, Support=166

Classification Report:
              precision    recall  f1-score   support

       maybe     0.0000    0.0000    0.0000        33
          no     0.3778    0.3366    0.3560       101
         yes     0.5721    0.7169    0.6364       166

    accuracy                         0.5100       300
   macro avg     0.3166    0.3512    0.3308       300
weighted avg     0.4438    0.5100    0.4720       300



### 5.4 Model 4: Random Forest

**Hypothesis**: Robust ensemble - good for feature importance, but may perform slightly worse than XGBoost.

In [12]:
rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'class_weight': ['balanced']
}

start_time = time.time()

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=-1),
    rf_params,
    cv=cv,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

rf_grid.fit(X_train, y_train)
training_time = time.time() - start_time

print(f"\n Best CV F1: {rf_grid.best_score_:.4f}")
print(f" Best params: {rf_grid.best_params_}")

rf_results = evaluate_model(
    "Random Forest",
    rf_grid.best_estimator_,
    X_test, y_test,
    training_time
)

all_results['Random Forest'] = rf_results

Fitting 5 folds for each of 24 candidates, totalling 120 fits

 Best CV F1: 0.3846
 Best params: {'class_weight': 'balanced', 'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}
Random Forest - Test Results
Accuracy:       0.5133
F1 (macro):     0.3439
F1 (weighted):  0.4814
Training time:  6.50s
Inference:      0.046 ms/sample

Per-class metrics:
  maybe   : P=0.0000, R=0.0000, F1=0.0000, Support=33
  no      : P=0.4019, R=0.4257, F1=0.4135, Support=101
  yes     : P=0.5751, R=0.6687, F1=0.6184, Support=166

Classification Report:
              precision    recall  f1-score   support

       maybe     0.0000    0.0000    0.0000        33
          no     0.4019    0.4257    0.4135       101
         yes     0.5751    0.6687    0.6184       166

    accuracy                         0.5133       300
   macro avg     0.3257    0.3648    0.3439       300
weighted avg     0.4535    0.5133    0.4814       300



### 5.5 Model 5: Multi-Layer Perceptron (Neural Network)

**Hypothesis**: Non-linearity may not help - TF-IDF features are already well-separated linearly.

In [13]:

mlp_params = {
    'hidden_layer_sizes': [(100,), (100, 50), (200, 100)],
    'activation': ['relu'],
    'alpha': [0.0001, 0.001],
    'learning_rate': ['adaptive'],
    'early_stopping': [True],
    'validation_fraction': [0.1]
}


start_time = time.time()

mlp_grid = GridSearchCV(
    MLPClassifier(random_state=RANDOM_SEED, max_iter=500),
    mlp_params,
    cv=cv,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

mlp_grid.fit(X_train, y_train)
training_time = time.time() - start_time

print(f"\n Best CV F1: {mlp_grid.best_score_:.4f}")
print(f" Best params: {mlp_grid.best_params_}")

mlp_results = evaluate_model(
    "MLP (Neural Network)",
    mlp_grid.best_estimator_,
    X_test, y_test,
    training_time
)

all_results['MLP'] = mlp_results



Fitting 5 folds for each of 6 candidates, totalling 30 fits

 Best CV F1: 0.2945
 Best params: {'activation': 'relu', 'alpha': 0.0001, 'early_stopping': True, 'hidden_layer_sizes': (100,), 'learning_rate': 'adaptive', 'validation_fraction': 0.1}
MLP (Neural Network) - Test Results
Accuracy:       0.5733
F1 (macro):     0.3386
F1 (weighted):  0.4949
Training time:  15.97s
Inference:      0.005 ms/sample

Per-class metrics:
  maybe   : P=0.0000, R=0.0000, F1=0.0000, Support=33
  no      : P=0.5366, R=0.2178, F1=0.3099, Support=101
  yes     : P=0.5792, R=0.9036, F1=0.7059, Support=166

Classification Report:
              precision    recall  f1-score   support

       maybe     0.0000    0.0000    0.0000        33
          no     0.5366    0.2178    0.3099       101
         yes     0.5792    0.9036    0.7059       166

    accuracy                         0.5733       300
   macro avg     0.3719    0.3738    0.3386       300
weighted avg     0.5011    0.5733    0.4949       300



## 6. Comprehensive Model Comparison

Now we compare all 5 models across multiple dimensions.

### 6.1 Overall Performance Comparison Table

In [14]:
comparison_data = []

for name, results in all_results.items():
    comparison_data.append({
        'Model': name,
        'Accuracy': results['accuracy'],
        'F1 (macro)': results['f1_macro'],
        'F1 (weighted)': results['f1_weighted'],
        'Training Time (s)': results['training_time'],
        'Inference (ms)': results['inference_time_ms']
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('F1 (macro)', ascending=False).reset_index(drop=True)

comparison_df.insert(0, 'Rank', range(1, len(comparison_df) + 1))

print("OVERALL PERFORMANCE COMPARISON")

pd.options.display.float_format = '{:.4f}'.format
print(comparison_df.to_string(index=False))

best_model = comparison_df.iloc[0]['Model']
best_f1 = comparison_df.iloc[0]['F1 (macro)']
print(f"\n BEST MODEL: {best_model} (F1 macro = {best_f1:.4f})")

OVERALL PERFORMANCE COMPARISON
 Rank               Model  Accuracy  F1 (macro)  F1 (weighted)  Training Time (s)  Inference (ms)
    1 Logistic Regression    0.5333      0.4296         0.5166             2.2410          0.0034
    2          Linear SVM    0.5433      0.3702         0.5054             0.3056          0.0027
    3       Random Forest    0.5133      0.3439         0.4814             6.4957          0.0460
    4                 MLP    0.5733      0.3386         0.4949            15.9660          0.0053
    5             XGBoost    0.5100      0.3308         0.4720            42.5345          0.0079

 BEST MODEL: Logistic Regression (F1 macro = 0.4296)


### 6.2 Per-Class Performance Comparison Table

In [15]:
per_class_data = []

for name, results in all_results.items():
    row = {'Model': name}
    for cls in ['yes', 'no', 'maybe']:
        row[f'F1 ({cls})'] = results['per_class_f1'][cls]
        row[f'Precision ({cls})'] = results['per_class_precision'][cls]
        row[f'Recall ({cls})'] = results['per_class_recall'][cls]
    per_class_data.append(row)

per_class_df = pd.DataFrame(per_class_data)

print("PER-CLASS F1 SCORES")


f1_cols = ['Model', 'F1 (yes)', 'F1 (no)', 'F1 (maybe)']
print(per_class_df[f1_cols].to_string(index=False))

print("\nBest model per class:")
for cls in ['yes', 'no', 'maybe']:
    best_idx = per_class_df[f'F1 ({cls})'].idxmax()
    best_model_cls = per_class_df.loc[best_idx, 'Model']
    best_f1_cls = per_class_df.loc[best_idx, f'F1 ({cls})']
    print(f"  {cls:8s}: {best_model_cls:25s} (F1 = {best_f1_cls:.4f})")
    
print("="*90)

PER-CLASS F1 SCORES
              Model  F1 (yes)  F1 (no)  F1 (maybe)
Logistic Regression    0.6369   0.4080      0.2439
         Linear SVM    0.6632   0.3934      0.0541
            XGBoost    0.6364   0.3560      0.0000
      Random Forest    0.6184   0.4135      0.0000
                MLP    0.7059   0.3099      0.0000

Best model per class:
  yes     : MLP                       (F1 = 0.7059)
  no      : Random Forest             (F1 = 0.4135)
  maybe   : Logistic Regression       (F1 = 0.2439)


### 6.3 Detailed Per-Class Metrics Table

In [16]:
print("DETAILED PER-CLASS METRICS (Precision, Recall, F1)")

per_class_df_sorted = per_class_df.copy()
per_class_df_sorted['Avg F1'] = per_class_df_sorted[['F1 (yes)', 'F1 (no)', 'F1 (maybe)']].mean(axis=1)
per_class_df_sorted = per_class_df_sorted.sort_values('Avg F1', ascending=False)

all_cols = ['Model', 
            'Precision (yes)', 'Recall (yes)', 'F1 (yes)',
            'Precision (no)', 'Recall (no)', 'F1 (no)',
            'Precision (maybe)', 'Recall (maybe)', 'F1 (maybe)']

print(per_class_df_sorted[all_cols].to_string(index=False))

DETAILED PER-CLASS METRICS (Precision, Recall, F1)
              Model  Precision (yes)  Recall (yes)  F1 (yes)  Precision (no)  Recall (no)  F1 (no)  Precision (maybe)  Recall (maybe)  F1 (maybe)
Logistic Regression           0.5938        0.6867    0.6369          0.4100       0.4059   0.4080             0.6250          0.1515      0.2439
         Linear SVM           0.5888        0.7590    0.6632          0.4390       0.3564   0.3934             0.2500          0.0303      0.0541
      Random Forest           0.5751        0.6687    0.6184          0.4019       0.4257   0.4135             0.0000          0.0000      0.0000
                MLP           0.5792        0.9036    0.7059          0.5366       0.2178   0.3099             0.0000          0.0000      0.0000
            XGBoost           0.5721        0.7169    0.6364          0.3778       0.3366   0.3560             0.0000          0.0000      0.0000


### 6.4 Statistical Summary

In [17]:
print("\n" + "="*70)
print("STATISTICAL SUMMARY")

best_model = comparison_df.iloc[0]['Model']
best_f1 = comparison_df.iloc[0]['F1 (macro)']
best_acc = comparison_df.iloc[0]['Accuracy']

worst_model = comparison_df.iloc[-1]['Model']
worst_f1 = comparison_df.iloc[-1]['F1 (macro)']

print(f"\n Best Model: {best_model}")
print(f"  Accuracy:   {best_acc:.4f}")
print(f"  F1 (macro): {best_f1:.4f}")

print(f"\n Worst Model: {worst_model}")
print(f"  F1 (macro): {worst_f1:.4f}")


gap = best_f1 - worst_f1
gap_pct = (gap / worst_f1) * 100
print(f"\n Performance Gap:")
print(f"  Absolute: {gap:.4f}")
print(f"  Relative: {gap_pct:.2f}%")


fastest_idx = comparison_df['Training Time (s)'].idxmin()
fastest_model = comparison_df.loc[fastest_idx, 'Model']
fastest_time = comparison_df.loc[fastest_idx, 'Training Time (s)']
fastest_f1 = comparison_df.loc[fastest_idx, 'F1 (macro)']

print(f"\n Fastest Training: {fastest_model}")
print(f"  Time: {fastest_time:.2f}s")
print(f"  F1 (macro): {fastest_f1:.4f}")

maybe_scores = {name: all_results[name]['per_class_f1']['maybe'] 
                for name in all_results.keys()}
best_maybe = max(maybe_scores.items(), key=lambda x: x[1])

print(f"\n Best on 'maybe' class: {best_maybe[0]}")
print(f"  F1 (maybe): {best_maybe[1]:.4f}")



STATISTICAL SUMMARY

 Best Model: Logistic Regression
  Accuracy:   0.5333
  F1 (macro): 0.4296

 Worst Model: XGBoost
  F1 (macro): 0.3308

 Performance Gap:
  Absolute: 0.0988
  Relative: 29.86%

 Fastest Training: Linear SVM
  Time: 0.31s
  F1 (macro): 0.3702

 Best on 'maybe' class: Logistic Regression
  F1 (maybe): 0.2439
